<a href="https://colab.research.google.com/github/Nikhilsankhyan0/Agrisheild-AI/blob/main/NLP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1.Importing Libraries
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# SpaCy
import spacy
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,ConfusionMatrixDisplay

# 2. Downloading NLTK Resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# 3. Load Dataset
nlp=spacy.load("en_core_web_sm")

# 4. load dataset
from google.colab import drive
drive.mount('/content/drive')

file_path="/content/drive/MyDrive/twitter_training.csv"
df=pd.read_csv(file_path,header=None)












[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

first five records:
   Tweet_ID       Entity Sentiment  \
0      2401  Borderlands  Positive   
1      2401  Borderlands  Positive   
2      2401  Borderlands  Positive   
3      2401  Borderlands  Positive   
4      2401  Borderlands  Positive   

                                                Text  
0  im getting on borderlands and i will murder yo...  
1  I am coming to the borders and I will kill you...  
2  im getting on borderlands and i will kill you ...  
3  im coming on borderlands and i will murder you...  
4  im getting on borderlands 2 and i will murder ...  

Dataset Shape:
(74682, 4)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Tweet_ID   74682 non-null  int64 
 1   Entity     7

In [ ]:
# 5.Give meaningful column names
df.columns=["Tweet_ID","Entity","Sentiment","Text"]
print("\nfirst five records:")
print(df.head())

# 6. Basic dataset information
print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nSentiment Distribution:")
print(df['Sentiment'].value_counts())


In [ ]:
# 7. Remove Missing Values
df=df.dropna(subset=['Text','Sentiment'])
print("\nDataset after removing missing values:")
print(df.shape)

# 8. Remove Irrelevant Class
df=df[df['Sentiment'].isin(['Positive','Negative','Neutral'])]
print("\nSentiment distribution after removing Irrelevant:")
print(df['Sentiment'].value_counts())

In [ ]:
# 9. Text Preprocessing function
stop_words=set(stopwords.words("english"))
lemmatizer=WordNetLemmatizer()
def clean_text(text):
  text=text.lower()
  text=re.sub(r"http\S+","",text)
  text=re.sub(r'@\W+','',text)
  text=re.sub(r'#','',text)
  text=re.sub(r'\d+','',text)
  text=text.translate(str.maketrans("","",string.punctuation))
  # remove extra space
  text=re.sub(r"\s+"," ",text).strip()
  # Tokenization
  words=text.split()
  words=[word for word in words
         if word not in stop_words]
  # Lemmitization
  words=[lemmatizer.lemmatize(word) for word in words]
  return ' '.join(words)

In [ ]:
# 10. Original and Cleaned Text
df['Clean_Text']=df["Text"].apply(clean_text)
print('Original and Cleaned Text:')
print(df[['Text','Clean_Text']].head())


# 11.Remove empty text records
df=df[df['Clean_Text'].str.strip()!=""]

# 12. Optical Spacy Preprocessing
# SpaCy is used here for additional linguistic processing.
# This function
def spacy_preprocess(text):
  doc=nlp(text)
  tokens=[]
  for token in doc:
    if (
        not token.is_stop
        and not token.is_punct
        and not token.is_space
    ):
      tokens.append(token.lemma_.lower())
  return " ".join(tokens)

# Apply spaCy preprocessing
df["Preprocessed_Text"]=df["Clean_Text"].apply(spacy_preprocess)

# Display preprocessed text
print("\nText after spaCy preprocessing:")
print(df[["Text","Preprocessed_Text"]].head(10))

In [ ]:
# 13. Define Features and Targets
X=df["Preprocessed_Text"]
y=df["Sentiment"]

# 14. Split data into training and testing sets
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
    )
print("\nTraining samples:",len(X_train))
print("Testing samples:",len(X_test))

In [ ]:
# 15. TF-IDF Feature Extraction
vectorizer=TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95
    )
# Learn vocabulary from training data
X_train_tfidf=vectorizer.fit_transform(X_train)

# transform testing data
X_test_tfidf=vectorizer.transform(X_test)

print("\nTF-IDF Training Shape:")
print("Shape of X_train_tfidf:",X_train_tfidf.shape)
print("\nTF-IDF Testing Shape:")
print("Shape of X_test_tfidf:",X_test_tfidf.shape)